# Create directories to hold permutation results
Populate each with:

- seed.txt: random seed used for perm
- parent_perm.txt: permuted indices of labels for parent
- self_perm.txt: permuted indices of labels for self

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.perm import write_dirs, ensure_dfs_match
import pandas as pd
import numpy as np

START_PERM = 0
END_PERM = 10  # non-inclusive
INCLUDE_BASELINE = True

Ensure all dfs for parent/self have matching indices/DAWBA IDs to ensure saved permuted indices will be consistent

In [2]:
def ensure_dfs_match(loop_dir_imp, loop_dir_shot, impact_type):
    all_dfs = []
    for model_dir in loop_dir_imp.iterdir():
        for seg_dir in model_dir.iterdir():
            file_path = seg_dir / f"{impact_type}.parquet"
            file = pd.read_parquet(file_path)
            assert (file.index == range(len(file))).all()
            all_dfs.append(file)
    for shot_dir in loop_dir_shot.iterdir():
        for model_dir in shot_dir.iterdir():
            for desc_dir in model_dir.iterdir():
                file_path = desc_dir / f"{impact_type}.parquet"
                file = pd.read_parquet(file_path)
                assert (file.index == range(len(file))).all()
                all_dfs.append(file)
    ## Ensure all DAWBA_IDs match
    ref = all_dfs[0]["DAWBA_ID"]
    return all(df["DAWBA_ID"].equals(ref) for df in all_dfs[1:])

In [3]:
loop_dir_imp = BASE_PATH / "data/w_probs"
loop_dir_shot = BASE_PATH / "data/w_0_shot_probs/temp_0.6"
assert ensure_dfs_match(loop_dir_imp, loop_dir_shot, "self")
assert ensure_dfs_match(loop_dir_imp, loop_dir_shot, "parent")

Create directories

In [4]:
# Arbitrary (bs of previous check) import for each of parent/self
imp_dir = BASE_PATH / "data/w_probs/Llama-3.3-8B-Instruct/pt_noshort"
self_df = pd.read_parquet(imp_dir / "self.parquet")
parent_df = pd.read_parquet(imp_dir / "parent.parquet")
write_dirs(
    start_perm=START_PERM,
    end_perm=END_PERM,
    self_df=self_df,
    parent_df=parent_df,
    save_dir=BASE_PATH / "perm_res" / "preds",
    start_seed=SEED,
    add_baseline=INCLUDE_BASELINE,
)

Check seeds

In [5]:
loop_dir = BASE_PATH / "perm_res" / "preds"
for perm_dir in loop_dir.iterdir():
    perm_name = perm_dir.name
    seed_file = perm_dir / "seed.txt"
    if perm_name == "baseline":
        assert not seed_file.exists()
    else:
        perm_num = int(perm_name.split("_")[1].strip(""))
        with open(seed_file, "r") as f:
            seed = int(f.read())
        assert seed == SEED + perm_num